## Scanpy Analysis ##

Uses a h5ad file


In [1]:
# Import python modules

import numpy as np
import pandas as pandas
import scanpy as sc
import anndata as ad
#import scirpy as ir 
#import muon as mu
#import skbio as sk
import os as os
#from muon import MuData
from matplotlib import pyplot as plt, cm as mpl_cm
#from cycler import cycler

#sc.set_figure_params(figsize=(6, 6))  ## generates an error, get this code for current versions if its needed.
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [2]:
sc.logging.print_header()

/users/medafisa/miniforge3/envs/vdj/lib/python3.12/site-packages/session_info2/__init__.py:124: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  and (v := getattr(pkg, "__version__", None))


Package,Version
numpy,2.1.3
pandas,2.2.3
scanpy,1.11.0
anndata,0.11.3
matplotlib,3.10.1
Component,Info
Python,"3.12.9 | packaged by conda-forge | (main, Mar 4 2025, 22:48:41) [GCC 13.3.0]"
OS,Linux-5.14.0-427.33.1.el9_4.x86_64-x86_64-with-glibc2.34
CPU,"64 logical CPU cores, x86_64"
GPU,"ID: 0, NVIDIA A2, Driver: 560.35.03, Memory: 15356 MiB"


# Get the data in

In [ ]:
# Get the data in
work_folder = "PATH/runFolders/Scanpy_AllData" # change path
out_folder = "PATH/outputs/Scanpy_AllData_tests" # change path
h5ad_file = "BCR_CLLremoved_AbSeq_RNA_scanpy.h5ad"


In [ ]:
adata_path = os.path.join(work_folder, h5ad_file)

adata = ad.io.read_h5ad(adata_path)

In [ ]:
adata.shape

In [ ]:
adata

In [ ]:
adata.layers

In [ ]:
adata.layers
# Drop layers - if this has been through Cellismo for file conversion and annotation, it generates a log10_1p_norm layer that messes with scanpy. Drop it so scanpy works.
del adata.layers["log10_1p_norm"]


In [ ]:
adata.layers

# Should now have no layers in it

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

# Filter the cells

In [ ]:
# Filter low quality cells. Note that the min genes has been reduced to 7 (scanpy recommended is 100) 
# because the targeted panel is expected to return a much smaller number of genes than WTA. 
# 7 is ythe number of genes used in a standard 'B cell' experimental cell type call (scanpy tutorial) that are represented in this panel. 
# Experimental cell type is already called by cellismo, and we do not need 10/10 scanpy genes to call cells in this case, can filter on cellismo annotation. 

sc.pp.filter_cells(adata, min_genes=7)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata.shape

In [ ]:
# Sample name consistency - Cellismo adds a 'sample_name' for some batched datasets. This messes up the analysis later, make sure they're all named the same thing.

# Multi-sample if there is different notatation between columns

adata.obs.rename(columns={"Sample_Name":"Sample_Tag_Name"}, inplace=True)

# Check my data object columns

adata.obs.columns

# Normalisation
Normalisation here is run on the whole dataset, because the CLL cells were part of the original experiment, and there may be amplification bias that needs to take them inot account before the dataset is used without them.

In [ ]:
adata.layers

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()
adata.X[3].data[:200]

In [ ]:
adata.layers

In [ ]:
adata.X.data[:200]

In [ ]:
# X and counts should be the same before normalisation
adata.layers["counts"].data[:200]


In [ ]:
# Normalizing to median total counts
# Gives a warning - known bug https://github.com/scverse/scanpy/issues/1333. Likely because my data is fairly homogenous with some low count cells. 
sc.pp.normalize_total(adata,
                      inplace=True)
# Logarithmize the data
sc.pp.log1p(adata)

In [ ]:
adata.layers

In [ ]:
print(adata.X[3].data[:200])

In [ ]:
# X should now be normalised and "counts" is still the raw data
adata.layers["counts"].data[:200]

# Batch correction
Only perform this at this point if the effect of batchy correction has already been checked in scanpy_analysis_AllData_tests.ipynb

In [ ]:
# Batch correction needs a key from obs, check obs keys
# This is likely to be 'original_sample' for cellismo integrated data. For now Sample_Tag_Name is used. 
#  Harmony https://github.com/immunogenomics/harmony is used by Cellismo, on command line usually uses a Seurat object and therefore is most compatible with R. It appears to over-correct when tested in cellismo. 
adata.obs_keys

In [ ]:
# This should = True. I've checked it and there is a change int he normalisation dot plot with 

sc.pp.combat(adata, key='Original Sample', inplace=True)

# Feature selection

In [ ]:
# Pick out the variable features between samples. 
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="Sample_Tag_Name")

In [ ]:
sc.pl.highly_variable_genes(adata)

# Dimensionality Reduction

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca_loadings(adata, components=(1, 2), include_lowest=True)

In [ ]:
sc.pl.pca(
    adata,
    color=["Sample_Tag_Name", "Sample_Tag_Name", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color="Sample_Tag_Name",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(
    adata,
    color="Original Sample",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

# Testing cell type by condition
Adjust these tests by what I would like to test

In [ ]:
# This version shold have an unassigned gtoup because I have not filtered out the cells unassigned to Bcell_type. 

sc.pl.umap(
    adata,
    color="BCell_type",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
# shape should be the same as previous. 
adata.shape

# Annotating the cell types by Leiden cluster method
AbSeq has the limitation of not annotating all the cells, especially if accuracy has been prioritised over inclusion of all cells in cellismo.

In [ ]:
# Leiden clustering umap
# This baseline one appears to generate Leiden res = 1.0

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
# Refine the clusters by adjusting the leiden resolution.

for res in [0.90]:
    sc.tl.leiden(adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")

In [ ]:
# Make sure this is 2 d.p. or it won't find it in obs. 

sc.pl.umap(
    adata,
    color=["leiden_res_0.90"],
    legend_loc="on data"
)

In [ ]:
# Save an image of the leiden res umap I want to use
FILE = os.path.join(out_folder, "umap_leiden_res_0.69.pdf")
sc.pl.umap(
    adata,
    color="leiden_res_0.69",
    legend_loc="on data",
    save=[FILE]
)

In [ ]:
adata.shape

# Make marker genes per cluster
Find the marker genes sets, because subtypes poorly described in literature (especially difference beytween switched and unswitched)
Use all adata, not just AbSeq annotated. 

In [ ]:
# Obtain cluster-specific differentially expressed genes
# Leiden must be to 2dp like the graph above
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0.69", method="wilcoxon")

In [ ]:
from matplotlib.pyplot import rc_context

FILE = os.path.join(out_folder, "DotPlot10.pdf")
#fig, ax=plt.subplots(figsize=(15,5))
# fig, ax=plt.subplots 
#(figsize=(15,15))

#with rc_context({"figure.figsize": (15, 5)}):
sc.pl.rank_genes_groups_dotplot(
    adata, 
    groupby="leiden_res_0.69", 
    standard_scale="var", 
    n_genes=5,
    save=[FILE]
    )

# Name the Leiden clusters

In [ ]:
adata.obs["cell_type_lvl1"] = adata.obs["leiden_res_0.69"].map(
    {
        "0": "Naive",
        "1": "Unswitched_memory",
        "2": "Naive",
        "3": "Activated",
        "4": "Unswitched_memory",
        "5": "Activated",
        "6": "Activated",
        "7": "Switched_memory",
        "8": "Switched_memory",
        "9": "Plasma_cells",
    }
)

# Get adata out as h5ad for cellismo or pydeseq2

In [ ]:
adata.layers

In [ ]:
# use raw counts
if hasattr(adata, 'raw') and adata.raw is not None:
    count_matrix = adata.raw.X
elif 'counts' in adata.layers:
    count_matrix = adata.layers['counts']
else:
    count_matrix = adata.X

In [ ]:
count_matrix

In [ ]:
FILE = os.path.join(out_folder, "ForDESeqV9.h5ad")

ad.io.write_h5ad(FILE, adata)

In [ ]:
adata.obs.head()

# Highly expressed genes

In [ ]:
sc.pl.highest_expr_genes(adata,
                         n_top=20)

In [ ]:
mask = ~adata.obs["BCell_type"].isin([#"Naive",
                                                 "Unswitched_Memory",
                                                 "ClassSwitched_Memory",
                                                 "DoubleNeg"
                                           ])
adata_BCellType = adata_group[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in adata_group.obs.columns:
    if pd.api.types.is_categorical_dtype(adata_group.obs[col].dtype):
        adata_group.obs[col] = adata_group.obs[col].cat.remove_unused_categories()

adata_groupA_Ctl_Naive = adata_group